# Phase 1 finalization: benchmark, error analysis, selection and inference
This notebook compares the three sentiment notebooks that we have saved their models and their validation prediction CSV files.

In [ ]:
# project setup
from pathlib import Path
import sys

PROJECT_ROOT = Path(
    "/content/voxforge-ai-review-analytics"
).resolve()

if not (PROJECT_ROOT / "src").exists():
    raise FileNotFoundError(
        f"Could not find src/ inside {PROJECT_ROOT}"
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%cd /content/voxforge-ai-review-analytics

print("Project root:", PROJECT_ROOT)

/content/voxforge-ai-review-analytics
Project root: /content/voxforge-ai-review-analytics


In [2]:
# connect google drive
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# copy the dataset from google drive
from pathlib import Path
import shutil

PROJECT_ROOT = Path("/content/voxforge-ai-review-analytics")

DRIVE_DATASET_PATH = Path(
    "/content/drive/MyDrive/VoxForge/data/processed/cleaned_reviews.csv"
)

PROJECT_DATASET_PATH = (
    PROJECT_ROOT / "data" / "processed" / "cleaned_reviews.csv"
)

PROJECT_DATASET_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

if not DRIVE_DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found in Google Drive: {DRIVE_DATASET_PATH}"
    )

shutil.copy2(
    DRIVE_DATASET_PATH,
    PROJECT_DATASET_PATH,
)

print("Dataset copied successfully.")
print("Source:", DRIVE_DATASET_PATH)
print("Destination:", PROJECT_DATASET_PATH)
print("Exists:", PROJECT_DATASET_PATH.exists())

Dataset copied successfully.
Source: /content/drive/MyDrive/VoxForge/data/processed/cleaned_reviews.csv
Destination: /content/voxforge-ai-review-analytics/data/processed/cleaned_reviews.csv
Exists: True


In [4]:
import json
import pandas as pd
from src.config import *
from src.data.load import load_csv
from src.sentiment.comparison import benchmark_from_predictions, build_benchmark_table, generate_comparison_report, path_size_mb
from src.sentiment.error_analysis import build_error_analysis, save_error_subsets, compare_two_models
from src.sentiment.model_selection import select_production_model, save_selection
from src.sentiment.inference import SentimentPredictor, enrich_reviews
from src.sentiment.benchmarking import benchmark_predictor, save_speed_benchmark
create_project_directories()

## 1. Register completed model artifacts and prediction files
Adjust only a path if your saved filename is different.

In [ ]:
MODEL_RUNS = [
    {"model_id": "sentiment_01", "model_name": "TF-IDF Logistic Regression", "artifact": SENTIMENT_MODELS_DIR / "tfidf_logistic_regression.joblib", "predictions": PREDICTIONS_DIR / "tfidf_logistic_validation.csv"},
    {"model_id": "sentiment_02", "model_name": "DistilBERT", "artifact": SENTIMENT_MODELS_DIR / "distilbert_sentiment", "predictions": PREDICTIONS_DIR / "distilbert_validation_predictions.csv"},
    {"model_id": "sentiment_03", "model_name": "DeBERTa v3", "artifact": SENTIMENT_MODELS_DIR / "deberta_v3_sentiment", "predictions": PREDICTIONS_DIR / "deberta_v3_sentiment_validation.csv"},
]
tracker = pd.read_csv(MODEL_TRACKING_PATH) if MODEL_TRACKING_PATH.exists() else pd.DataFrame()
MODEL_RUNS

[{'model_id': 'sentiment_01',
  'model_name': 'TF-IDF Logistic Regression',
  'artifact': PosixPath('/content/voxforge-ai-review-analytics/models/sentiment/tfidf_logistic_regression.joblib'),
  'predictions': PosixPath('/content/voxforge-ai-review-analytics/results/predictions/tfidf_logistic_validation.csv')},
 {'model_id': 'sentiment_02',
  'model_name': 'DistilBERT',
  'artifact': PosixPath('/content/voxforge-ai-review-analytics/models/sentiment/distilbert_sentiment'),
  'predictions': PosixPath('/content/voxforge-ai-review-analytics/results/predictions/distilbert_validation_predictions.csv')},
 {'model_id': 'sentiment_03',
  'model_name': 'DeBERTa v3',
  'artifact': PosixPath('/content/voxforge-ai-review-analytics/models/sentiment/deberta_v3_sentiment'),
  'predictions': PosixPath('/content/voxforge-ai-review-analytics/results/predictions/deberta_v3_sentiment_validation.csv')}]

In [13]:
rows = []
loaded_predictions = {}
for item in MODEL_RUNS:
    if not item["predictions"].exists():
        print("Skipping missing predictions:", item["predictions"])
        continue
    predictions = pd.read_csv(item["predictions"])
    loaded_predictions[item["model_id"]] = predictions
    tracked = tracker.loc[tracker["model_id"] == item["model_id"]].tail(1) if not tracker.empty else pd.DataFrame()
    rows.append(benchmark_from_predictions(
        predictions, model_id=item["model_id"], model_name=item["model_name"],
        training_time_seconds=None if tracked.empty else tracked.iloc[0].get("training_time_seconds"),
        inference_time_ms=None if tracked.empty else tracked.iloc[0].get("inference_time_ms"),
        model_size_mb=path_size_mb(item["artifact"]),
    ))
benchmark = build_benchmark_table(rows, MODEL_BENCHMARK_PATH)
display(benchmark)

Skipping missing predictions: /content/voxforge-ai-review-analytics/results/predictions/distilbert_validation_predictions.csv


,model_id,model_name,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,training_time_seconds,average_inference_ms,model_size_mb,validation_rows,negative_precision,negative_recall,negative_f1,neutral_precision,neutral_recall,neutral_f1,positive_precision,positive_recall,positive_f1
0,sentiment_01,TF-IDF Logistic Regression,0.900978,0.610413,0.733390,0.655143,0.914125,2.867864,0.034427,1.853779,7564,0.572539,0.749153,0.649046,0.279294,0.52568,0.36478,0.979405,0.925339,0.951605
1,sentiment_03,DeBERTa v3,0.038863,0.012954,0.333333,0.024940,0.002908,NaN,NaN,1918.702287,7565,0.038863,1.000000,0.074819,0.000000,0.00000,0.00000,0.000000,0.000000,0.000000


## 2. Error analysis

In [14]:
for item in MODEL_RUNS:
    if item["model_id"] not in loaded_predictions:
        continue
    analysis = build_error_analysis(loaded_predictions[item["model_id"]])
    save_error_subsets(analysis, ERROR_ANALYSIS_DIR, item["model_id"])
    display(analysis.loc[~analysis["is_correct"]].head(10))

/content/voxforge-ai-review-analytics/src/sentiment/error_analysis.py:22: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  frame["contains_contrast"] = frame["text"].str.contains(


,text,true_label,predicted_label,probability_negative,probability_neutral,probability_positive,is_correct,error_type,text_length,word_count,contains_contrast,confidence
3,decent tablet good price one issue starting is...,positive,neutral,0.060043,0.774972,0.164985,False,positive -> neutral,143,23,False,0.774972
6,got battery wireless mouse got battery wireles...,positive,negative,0.767681,0.145840,0.086480,False,positive -> negative,116,18,False,0.767681
10,wonderful starter child bought one year old so...,positive,neutral,0.033637,0.663130,0.303232,False,positive -> neutral,155,25,False,0.663130
24,awesome enjoy small device well many feature o...,positive,neutral,0.099628,0.581703,0.318669,False,positive -> neutral,118,19,False,0.581703
26,durable great kid durable nice download softwa...,positive,neutral,0.162561,0.478467,0.358972,False,positive -> neutral,72,11,False,0.478467
45,oh yeah long lasting cheap battery ent,positive,negative,0.547252,0.185099,0.267650,False,positive -> negative,38,7,False,0.547252
46,sure battery fine mind packaging probably buy ...,negative,neutral,0.349137,0.428019,0.222844,False,negative -> neutral,138,20,False,0.428019
55,charger charger supposed,neutral,negative,0.558154,0.120268,0.321577,False,neutral -> negative,24,3,False,0.558154
56,switched duracell lately aa aaa battery bought...,positive,negative,0.401452,0.337929,0.260619,False,positive -> negative,1314,201,False,0.401452
65,inexpensive access learn use apps sure enjoy t...,neutral,positive,0.051026,0.364665,0.584309,False,neutral -> positive,62,9,False,0.584309


/content/voxforge-ai-review-analytics/src/sentiment/error_analysis.py:22: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  frame["contains_contrast"] = frame["text"].str.contains(


,text,true_label,predicted_label,probability_negative,probability_neutral,probability_positive,is_correct,error_type,text_length,word_count,contains_contrast,confidence
0,great for audible books Needed for a friend th...,positive,negative,NaN,NaN,NaN,False,positive -> negative,92,18,False,NaN
1,Love it Works great with my Ring doorbell and ...,positive,negative,NaN,NaN,NaN,False,positive -> negative,115,23,False,NaN
2,Great This is a LOT of batteries for the price...,positive,negative,NaN,NaN,NaN,False,positive -> negative,213,40,False,NaN
3,Excellent product!!! Excellent product!!! Exce...,positive,negative,NaN,NaN,NaN,False,positive -> negative,90,9,False,NaN
4,Great first tablet! This is a great tablet for...,positive,negative,NaN,NaN,NaN,False,positive -> negative,97,20,False,NaN
5,High Value Gift This tablet was a gift for my ...,positive,negative,NaN,NaN,NaN,False,positive -> negative,526,97,False,NaN
6,Good value for money My first amazon tablet do...,positive,negative,NaN,NaN,NaN,False,positive -> negative,331,63,False,NaN
7,Can't get better This is the best entry level ...,positive,negative,NaN,NaN,NaN,False,positive -> negative,205,39,False,NaN
8,Amazon Echo As the community grows and more sk...,positive,negative,NaN,NaN,NaN,False,positive -> negative,299,59,True,NaN
9,Easy to set up Very easy to set up. I mostly c...,positive,negative,NaN,NaN,NaN,False,positive -> negative,300,63,False,NaN


In [15]:
if "sentiment_02" in loaded_predictions and "sentiment_03" in loaded_predictions:
    comparisons = compare_two_models(loaded_predictions["sentiment_02"], loaded_predictions["sentiment_03"], first_name="distilbert", second_name="deberta")
    for name, frame in comparisons.items():
        frame.to_csv(ERROR_ANALYSIS_DIR / f"{name}.csv", index=False)
        print(name, len(frame))

## 3. Select the production model and write the report

In [17]:
winner, reason = select_production_model(benchmark)
selected_run = next(item for item in MODEL_RUNS if item["model_id"] == winner["model_id"])
save_selection(winner, reason, artifact_path=selected_run["artifact"], output_path=MODEL_SELECTION_PATH)
generate_comparison_report(benchmark, selected_model=winner["model_name"], reason=reason, output_path=MODEL_COMPARISON_REPORT_PATH)
print("Winner:", winner["model_name"])
print(reason)

Winner: TF-IDF Logistic Regression
Selected the highest Macro F1 because both transformer candidates were not available.


## 4. Speed benchmark for the selected model

In [18]:
reviews = load_csv(CLEANED_REVIEWS_PATH)
text_column = "transformer_text" if selected_run["artifact"].is_dir() else "classical_text"
predictor = SentimentPredictor(selected_run["artifact"])
speed = benchmark_predictor(predictor, reviews[text_column].dropna().head(1000), model_name=winner["model_name"], batch_sizes=(1, 8, 32))
save_speed_benchmark([speed], INFERENCE_SPEED_PATH)
display(speed)

,model_name,device,batch_size,sample_count,timed_runs,total_seconds,reviews_per_second,average_ms_per_review,python_version,torch_version,gpu_name
0,TF-IDF Logistic Regression,cpu,1,100,3,0.009415,10621.635088,0.094147,3.12.13,2.11.0+cu128,Tesla T4
1,TF-IDF Logistic Regression,cpu,8,100,3,0.009013,11095.052461,0.090130,3.12.13,2.11.0+cu128,Tesla T4
2,TF-IDF Logistic Regression,cpu,32,320,3,0.020171,15864.296281,0.063035,3.12.13,2.11.0+cu128,Tesla T4


## 5. Run production inference on the complete cleaned dataset

In [19]:
enriched = enrich_reviews(reviews, predictor, text_column=text_column, output_path=ENRICHED_REVIEWS_PATH)
print(f"Saved {len(enriched):,} enriched reviews to {ENRICHED_REVIEWS_PATH}")
display(enriched.head())

Saved 47,279 enriched reviews to /content/voxforge-ai-review-analytics/data/processed/reviews_with_sentiment.csv


,product_name,asins,brand,categories,review_date,rating,review_text,review_title,source_dataset,primary_categories,sentiment_label,normalized_combined_text,transformer_text,classical_text,predicted_sentiment,probability_negative,probability_neutral,probability_positive,sentiment_confidence
0,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",2017-01-13 00:00:00+00:00,5,This product so far has not disappointed. My c...,Kindle,cleaned_reviews,NaN,positive,Kindle This product so far has not disappointe...,Kindle This product so far has not disappointe...,kindle product far not disappointed child love...,positive,0.081628,0.107068,0.811305,0.811305
1,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",2017-01-13 00:00:00+00:00,5,great for beginner or experienced person. Boug...,very fast,cleaned_reviews,NaN,positive,very fast great for beginner or experienced pe...,very fast great for beginner or experienced pe...,fast great beginner experienced person bought ...,positive,0.021511,0.311388,0.667101,0.667101
2,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",2017-01-13 00:00:00+00:00,5,Inexpensive tablet for him to use and learn on...,Beginner tablet for our 9 year old son.,cleaned_reviews,NaN,positive,Beginner tablet for our 9 year old son. Inexpe...,Beginner tablet for our 9 year old son. Inexpe...,beginner tablet year old son inexpensive table...,neutral,0.027128,0.560272,0.412599,0.560272
3,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",2017-01-13 00:00:00+00:00,4,I've had my Fire HD 8 two weeks now and I love...,Good!!!,cleaned_reviews,NaN,positive,Good!!! I've had my Fire HD 8 two weeks now an...,Good!!! I've had my Fire HD 8 two weeks now an...,good fire hd two week love tablet great value ...,positive,0.002640,0.050085,0.947275,0.947275
4,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",B01AHB9CN2,Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",2017-01-12 00:00:00+00:00,5,I bought this for my grand daughter when she c...,Fantastic Tablet for kids,cleaned_reviews,NaN,positive,Fantastic Tablet for kids I bought this for my...,Fantastic Tablet for kids I bought this for my...,fantastic tablet kid bought grand daughter com...,positive,0.010699,0.062902,0.926399,0.926399


In [ ]:
# store the dataset in the from colab drive to local machine
from google.colab import files

files.download(
    "/content/voxforge-ai-review-analytics/data/processed/reviews_with_sentiment.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Ablation study note
The reusable tracker is in `src/sentiment/ablation.py`. Run only controlled retraining experiments that are useful: class weights on/off and max length 128/256. Do not block Phase 2 on ablations.